**DATA PROCESSING**

In [167]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

**DATA FIRST LOOK**

In [168]:
# 1. Load dataset
rawData = pd.read_csv("../data/11_obesity_risk_levels_train.csv")
print("Original dataset:", rawData.shape)
rawData.columns = rawData.columns.str.strip() #stripping whitespace

print("Column data types:")
print(rawData.dtypes)


Original dataset: (1794, 17)
Column data types:
Gender                                str
Age                               float64
Height                            float64
Weight                            float64
family_history_with_overweight        str
FAVC                                  str
FCVC                              float64
NCP                               float64
CAEC                                  str
SMOKE                                 str
CH2O                              float64
SCC                                   str
FAF                               float64
TUE                               float64
CALC                                  str
MTRANS                                str
NObeyesdad                            str
dtype: object


In [169]:
print("First 5 rows of the dataset:")
rawData.head()

First 5 rows of the dataset:


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Male,29.000000,1.666710,89.048151,yes,yes,1.897796,3.00000,Sometimes,no,1.193039,no,0.011519,0.000000,Sometimes,Automobile,Obesity_Type_I
1,Male,18.000000,1.820930,108.742005,yes,yes,2.000000,1.25535,Sometimes,no,2.497065,no,1.000000,1.441605,no,Public_Transportation,Obesity_Type_I
2,Male,21.727380,1.783782,105.000276,yes,yes,2.044326,3.00000,Sometimes,no,2.358180,no,0.349371,0.000000,Sometimes,Public_Transportation,Obesity_Type_I
3,Female,23.664709,1.738836,133.472641,yes,yes,3.000000,3.00000,Sometimes,no,2.863513,no,1.026452,0.714137,Sometimes,Public_Transportation,Obesity_Type_III
4,Female,21.000000,1.600000,61.000000,no,yes,2.000000,3.00000,Sometimes,no,1.000000,no,1.000000,1.000000,Sometimes,Public_Transportation,Normal_Weight


**CHECKING FOR MISSING AND DUPLICATE**

In [170]:
# 2. Check for missing values
print("\nMissing values:")
print(rawData.isnull().sum())



Missing values:
Gender                            0
Age                               0
Height                            0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                             0
CH2O                              0
SCC                               0
FAF                               0
TUE                               0
CALC                              0
MTRANS                            0
NObeyesdad                        0
dtype: int64


The dataset has 0 missing value

In [171]:
# 3. Check for and remove duplicates
print("\nDuplicate rows:", rawData.duplicated().sum())
processedData = rawData.drop_duplicates()
print("Dataset after removing duplicates:", processedData.shape)


Duplicate rows: 20
Dataset after removing duplicates: (1774, 17)


The dataset contains 20 duplicate rows out of 1,794 samples. Since these are exact duplicates, we removed them before training to avoid giving extra weight to repeated observations. The cleaned dataset contains 1,774 samples.

In [172]:
print("\nDuplicate rows:", processedData.duplicated().sum())


Duplicate rows: 0


Confirmed that no duplicates in the processedData

**CHECK NUMERIC IS VALID**

In [173]:
numeric_cols = processedData.select_dtypes(include=['number']).columns # Select numeric columns
print("Number of numeric columns:", len(numeric_cols))
print("Numeric columns data types:")
print(processedData[numeric_cols].dtypes)
print("Numeric columns summary:")
print(processedData[numeric_cols].describe())

print("Negative values before cleaning:")
print((processedData[numeric_cols] < 0).sum())

positive_cols = ["Age", "Height", "Weight"]
for col in positive_cols:
    processedData.loc[processedData[col] <= 0, col] = np.nan

for col in numeric_cols:
    processedData.loc[processedData[col] < 0, col] = np.nan

print("Missing values after invalid-value checks:")
print(processedData[numeric_cols].isnull().sum())
processedData = processedData.dropna(subset=numeric_cols)
print("Dataset after removing invalid rows:", processedData.shape)

Number of numeric columns: 8
Numeric columns data types:
Age       float64
Height    float64
Weight    float64
FCVC      float64
NCP       float64
CH2O      float64
FAF       float64
TUE       float64
dtype: object
Numeric columns summary:
               Age       Height       Weight         FCVC          NCP  \
count  1774.000000  1774.000000  1774.000000  1774.000000  1774.000000   
mean     24.289407     1.701380    86.677376     2.411621     2.698085   
std       6.302228     0.092217    26.123159     0.535862     0.758925   
min      14.000000     1.480000    39.101805     1.000000     1.000000   
25%      19.786243     1.630089    65.448643     2.000000     2.704076   
50%      22.838055     1.700010    83.111456     2.360207     3.000000   
75%      26.000000     1.767286   107.338764     3.000000     3.000000   
max      56.000000     1.980000   165.057269     3.000000     4.000000   

              CH2O          FAF          TUE  
count  1774.000000  1774.000000  1774.000000  

The numeric columns were checked for invalid values, and invalid rows were removed.

In [174]:
# 5. Fix inconsistent text formatting
categorical_cols = processedData.select_dtypes(include=['str']).columns.tolist()

for col in categorical_cols:
    processedData[col] = processedData[col].str.strip()
print("Number of categorical columns:", len(categorical_cols))
print("Categorical columns data types:")
print(processedData[categorical_cols].dtypes)
print("Categorical columns summary:")
print(processedData[categorical_cols].describe())



Number of categorical columns: 9
Categorical columns data types:
Gender                            str
family_history_with_overweight    str
FAVC                              str
CAEC                              str
SMOKE                             str
SCC                               str
CALC                              str
MTRANS                            str
NObeyesdad                        str
dtype: object
Categorical columns summary:
       Gender family_history_with_overweight  FAVC       CAEC SMOKE   SCC  \
count    1774                           1774  1774       1774  1774  1774   
unique      2                              2     2          4     2     2   
top      Male                            yes   yes  Sometimes    no    no   
freq      888                           1451  1577       1498  1741  1696   

             CALC                 MTRANS      NObeyesdad  
count        1774                   1774            1774  
unique          4                      5      

Modify NObeyesdad so that it is able for binary classification

In [175]:
# View the NObeyesdad unique values
print(processedData['NObeyesdad'].unique())

<StringArray>
[     'Obesity_Type_I',    'Obesity_Type_III',       'Normal_Weight',
 'Insufficient_Weight',     'Obesity_Type_II', 'Overweight_Level_II',
  'Overweight_Level_I']
Length: 7, dtype: str


In [176]:
obesity_labels = ["Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III"]
# Convert it into is_obesity column
processedData['is_obesity'] = processedData['NObeyesdad'].isin(obesity_labels).astype(int)

print(processedData['is_obesity'].value_counts())


is_obesity
0    949
1    825
Name: count, dtype: int64


In [177]:
# Drop the original NObeyesdad column
processedData = processedData.drop(columns=['NObeyesdad'])
print(processedData.head())

   Gender        Age    Height      Weight family_history_with_overweight  \
0    Male  29.000000  1.666710   89.048151                            yes   
1    Male  18.000000  1.820930  108.742005                            yes   
2    Male  21.727380  1.783782  105.000276                            yes   
3  Female  23.664709  1.738836  133.472641                            yes   
4  Female  21.000000  1.600000   61.000000                             no   

  FAVC      FCVC      NCP       CAEC SMOKE      CH2O SCC       FAF       TUE  \
0  yes  1.897796  3.00000  Sometimes    no  1.193039  no  0.011519  0.000000   
1  yes  2.000000  1.25535  Sometimes    no  2.497065  no  1.000000  1.441605   
2  yes  2.044326  3.00000  Sometimes    no  2.358180  no  0.349371  0.000000   
3  yes  3.000000  3.00000  Sometimes    no  2.863513  no  1.026452  0.714137   
4  yes  2.000000  3.00000  Sometimes    no  1.000000  no  1.000000  1.000000   

        CALC                 MTRANS  is_obesity  
0  Som

Before using one-hot encoded, we see that some columns are yes/no. Then change it to 1/0, this will reduce the number of features in training.


In [178]:
print(processedData['Gender'].value_counts())
print()

binary_cols = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_cols:
    print(processedData[col].value_counts())
    print()

Gender
Male      888
Female    886
Name: count, dtype: int64

family_history_with_overweight
yes    1451
no      323
Name: count, dtype: int64

FAVC
yes    1577
no      197
Name: count, dtype: int64

SMOKE
no     1741
yes      33
Name: count, dtype: int64

SCC
no     1696
yes      78
Name: count, dtype: int64



The dataset has a balanced gender distribution, with an equal number of male and female observations. Therefore, there is no gender imbalance.

In [179]:
print(processedData['Gender'].value_counts())
print()
for col in binary_cols:
    print(processedData[col].value_counts())
    print()

Gender
Male      888
Female    886
Name: count, dtype: int64

family_history_with_overweight
yes    1451
no      323
Name: count, dtype: int64

FAVC
yes    1577
no      197
Name: count, dtype: int64

SMOKE
no     1741
yes      33
Name: count, dtype: int64

SCC
no     1696
yes      78
Name: count, dtype: int64



In [180]:
print(processedData.dtypes)

Gender                                str
Age                               float64
Height                            float64
Weight                            float64
family_history_with_overweight        str
FAVC                                  str
FCVC                              float64
NCP                               float64
CAEC                                  str
SMOKE                                 str
CH2O                              float64
SCC                                   str
FAF                               float64
TUE                               float64
CALC                                  str
MTRANS                                str
is_obesity                          int64
dtype: object


**CHANGE CATEGORICAL TO NUMERIC**

In [181]:
categorical_cols = processedData.select_dtypes(include=["str"]).columns.to_list()
print(categorical_cols)

#   Using one-hot encoding for categorical columns
processedData = pd.get_dummies(processedData, columns=categorical_cols)

# Convert one-hot boolean columns to numeric 0/1 values.
bool_cols = processedData.select_dtypes(include=["bool"]).columns
processedData[bool_cols] = processedData[bool_cols].astype(int)

['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']


In [182]:
print(processedData.dtypes)

Age                                   float64
Height                                float64
Weight                                float64
FCVC                                  float64
NCP                                   float64
CH2O                                  float64
FAF                                   float64
TUE                                   float64
is_obesity                              int64
Gender_Female                           int64
Gender_Male                             int64
family_history_with_overweight_no       int64
family_history_with_overweight_yes      int64
FAVC_no                                 int64
FAVC_yes                                int64
CAEC_Always                             int64
CAEC_Frequently                         int64
CAEC_Sometimes                          int64
CAEC_no                                 int64
SMOKE_no                                int64
SMOKE_yes                               int64
SCC_no                            

In [183]:
# Compare shape with original data
print("Original shape:", rawData.shape)
print("Processed shape:", processedData.shape)

Original shape: (1794, 17)
Processed shape: (1774, 32)


The processed data has 32 columns (31 features and 1 label)

In [184]:
# Feature columns
target_col = 'is_obesity'
feature_cols = processedData.drop(columns=[target_col]).columns.to_list()
print(feature_cols)

# Label columns
label_cols = [target_col]
print(label_cols)

['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'Gender_Female', 'Gender_Male', 'family_history_with_overweight_no', 'family_history_with_overweight_yes', 'FAVC_no', 'FAVC_yes', 'CAEC_Always', 'CAEC_Frequently', 'CAEC_Sometimes', 'CAEC_no', 'SMOKE_no', 'SMOKE_yes', 'SCC_no', 'SCC_yes', 'CALC_Always', 'CALC_Frequently', 'CALC_Sometimes', 'CALC_no', 'MTRANS_Automobile', 'MTRANS_Bike', 'MTRANS_Motorbike', 'MTRANS_Public_Transportation', 'MTRANS_Walking']
['is_obesity']


In [185]:
# X is the feature matrix
X = processedData[feature_cols]
print(X.head())

# y is the label vector
y = processedData[target_col]
print(y.head())

         Age    Height      Weight      FCVC      NCP      CH2O       FAF  \
0  29.000000  1.666710   89.048151  1.897796  3.00000  1.193039  0.011519   
1  18.000000  1.820930  108.742005  2.000000  1.25535  2.497065  1.000000   
2  21.727380  1.783782  105.000276  2.044326  3.00000  2.358180  0.349371   
3  23.664709  1.738836  133.472641  3.000000  3.00000  2.863513  1.026452   
4  21.000000  1.600000   61.000000  2.000000  3.00000  1.000000  1.000000   

        TUE  Gender_Female  Gender_Male  ...  SCC_yes  CALC_Always  \
0  0.000000              0            1  ...        0            0   
1  1.441605              0            1  ...        0            0   
2  0.000000              0            1  ...        0            0   
3  0.714137              1            0  ...        0            0   
4  1.000000              1            0  ...        0            0   

   CALC_Frequently  CALC_Sometimes  CALC_no  MTRANS_Automobile  MTRANS_Bike  \
0                0               1   

**Now we split the data (80/20) - this can be changed if preferred**

In [186]:
# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# No separate validation set is created in this notebook.

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (1419, 31)
Testing: (355, 31)


**SPLIT TO TRAIN AND TEST**


In [187]:
# 8. Save the training dataset

trainData = X_train.copy()
trainData[target_col] = y_train
trainData.to_csv("../data/obesity_train.csv", index=False)

print("\nTraining dataset:", trainData.shape)
print("Training data saved successfully!")

# 9. Save the testing dataset
testData = X_test.copy()
testData[target_col] = y_test
testData.to_csv("../data/obesity_test.csv", index=False)

print("\nTesting dataset:", testData.shape)
print("Testing data saved successfully!")


Training dataset: (1419, 32)
Training data saved successfully!

Testing dataset: (355, 32)
Testing data saved successfully!
